In [1]:
# imports for EDA 
import pandas as pd
import numpy as  np
import matplotlib.pyplot as plt
import seaborn as sns
import re


In [ ]:
test_data = pd.read_csv('test.csv')
test_labels = pd.read_csv('test_labels.csv')

test_data.head()


,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [9]:
test_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   id            153164 non-null  str  
 1   comment_text  153164 non-null  str  
dtypes: str(2)
memory usage: 2.3 MB


In [11]:
test_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             153164 non-null  str  
 1   toxic          153164 non-null  int64
 2   severe_toxic   153164 non-null  int64
 3   obscene        153164 non-null  int64
 4   threat         153164 non-null  int64
 5   insult         153164 non-null  int64
 6   identity_hate  153164 non-null  int64
dtypes: int64(6), str(1)
memory usage: 8.2 MB


In [3]:
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# replace any -1 in labels with 0 (indicating absence of that label)
test_labels[label_cols] = test_labels[label_cols].replace(-1, 0)

In [7]:
test_labels.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0,0,0,0,0,0
1,0000247867823ef7,0,0,0,0,0,0
2,00013b17ad220c46,0,0,0,0,0,0
3,00017563c3f7919a,0,0,0,0,0,0
4,00017695ad8997eb,0,0,0,0,0,0


In [14]:
# 2. Merge text and labels on 'id'
df_test = pd.merge(test_data, test_labels, on='id')

# 3. Filter out all ignored rows (-1)
# Checking one label (e.g., 'toxic != -1') filters out all unannotated rows
df_test_clean = df_test[df_test['toxic'] != -1].reset_index(drop=True)

print(f"Original test rows : {len(df_test):,}")
print(f"Evaluated test rows: {len(df_test_clean):,}")

Original test rows : 153,164
Evaluated test rows: 63,978


In [15]:
df_test_clean.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,63978.000000,63978.000000,63978.000000,63978.000000,63978.000000,63978.000000
mean,0.095189,0.005736,0.057692,0.003298,0.053565,0.011129
std,0.293478,0.075522,0.233161,0.057334,0.225160,0.104905
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [17]:
df_test_clean.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0


In [18]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # 1. Replace line breaks and tabs with a single space
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 2. Remove IP addresses (common in Wikipedia edits)
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Standardize contraction apostrophes
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'d", " would", text)
    
    # 5. Keep only alphanumeric characters and basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s!\?]', '', text)
    
    # 6. Collapse multiple whitespaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [21]:
# apply the clean function on df_test_clean data

df_test_clean['clean_text'] = df_test_clean['comment_text'].apply(clean_text)

# then drop id column and comment_text column from df_test_clean
df_test_clean = df_test_clean.drop(columns=['id', 'comment_text'])

In [23]:
# Reorder columns: description first, then labels
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df_test_clean = df_test_clean[['clean_text'] + label_cols]

In [24]:
df_test_clean.head()

,clean_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,thank you for understanding i think very highl...,0,0,0,0,0,0
1,dear god this site is horrible,0,0,0,0,0,0
2,somebody will invariably try to add religion? ...,0,0,0,0,0,0
3,it says it right there that it is a type the t...,0,0,0,0,0,0
4,before adding a new product to the list make s...,0,0,0,0,0,0


In [25]:
# save the cleaned test data to a new CSV file
df_test_clean.to_csv('test_cleaned.csv', index=False)